# Week 1, Day 3: structured outputs and tool calling

Today's idea: a model may only *ask* for an action. **Your code** checks the request and runs it. The project built a task assistant with create, list, update and delete tools, and a confirmation before any delete. Here the same loop runs in Python with a stand-in model, so the checks around it are real and you can break them on purpose.

Run the cells from the top, one at a time (Shift+Enter).

## Step 1: a schema and a validator
The schema lists each tool's fields and types. **Every** request is validated before anything runs.

In [ ]:
import json, datetime
TOOLS = {
  "create_task": {"title": str, "due": str, "priority": str},
  "list_tasks":  {},
  "mark_done":   {"title": str},
  "delete_task": {"title": str},
}
PRIORITIES = {"low", "normal", "high"}

def validate(name, args):
    if name not in TOOLS: return f"unknown tool {name!r}"
    spec = TOOLS[name]
    extra = set(args) - set(spec)
    if extra: return f"unexpected fields {sorted(extra)}"
    for field, typ in spec.items():
        if field not in args: return f"missing field {field!r}"
        if not isinstance(args[field], typ): return f"{field!r} must be {typ.__name__}"
    if name == "create_task":
        if not 1 <= len(args["title"]) <= 80: return "title must be 1 to 80 characters"
        if args["priority"] not in PRIORITIES: return "priority must be low, normal or high"
        try: datetime.date.fromisoformat(args["due"])
        except ValueError: return "due must be a date like 2026-12-01"
    return None

print(validate("create_task", {"title": "Pay rent", "due": "2026-12-01", "priority": "high"}))
print(validate("create_task", {"title": "Pay rent", "due": "tomorrow", "priority": "high"}))
print(validate("drop_database", {}))

## Step 2: tools that run only in your code, for one person
Every tool takes the **user id** from your code, never from the model. Delete does not delete: it creates a *pending confirmation*.

In [ ]:
DB, PENDING, LOG = {}, {}, []      # tasks per user, confirmations waiting, an audit log

def run_tool(user, name, args):
    err = validate(name, args)
    LOG.append({"user": user, "tool": name, "ok": err is None, "error": err})        # every call is logged, valid or not
    if err: return {"error": err}
    tasks = DB.setdefault(user, [])
    if name == "create_task": tasks.append({**args, "done": False}); return {"created": args["title"]}
    if name == "list_tasks":  return {"tasks": tasks}
    if name == "mark_done":
        t = next((t for t in tasks if t["title"].lower() == args["title"].lower()), None)
        if not t: return {"error": "no such task"}
        t["done"] = True; return {"done": t["title"]}
    if name == "delete_task":
        if not any(t["title"].lower() == args["title"].lower() for t in tasks): return {"error": "no such task"}
        PENDING[user] = args["title"]; return {"confirm": f"Delete {args['title']!r}? Answer yes to confirm."}

def confirm(user, answer):
    title = PENDING.pop(user, None)
    if title is None or answer.lower() != "yes": return {"cancelled": True}
    DB[user] = [t for t in DB[user] if t["title"].lower() != title.lower()]
    return {"deleted": title}

print(run_tool("alice", "create_task", {"title": "Pay rent", "due": "2026-12-01", "priority": "high"}))
print(run_tool("alice", "delete_task", {"title": "Pay rent"}))
print("still there:", [t["title"] for t in DB["alice"]])
print(confirm("alice", "yes"), "| left:", DB["alice"])

## Step 3: the loop, and bad requests
A stand-in model turns a sentence into a tool request. Then see what happens when the model makes a mistake or an attacker tries something. Your code refuses each one and keeps going.

In [ ]:
def stand_in_model(sentence):
    s = sentence.lower()
    if s.startswith("add"): return ("create_task", {"title": sentence[4:].strip().capitalize(), "due": "2026-12-01", "priority": "normal"})
    if "my tasks" in s:      return ("list_tasks", {})
    if s.startswith("delete everything"): return ("delete_task", {"title": "*", "all_users": True})     # a bad request
    return ("drop_database", {})                                                                          # a request that must never run

for sentence in ["add buy milk", "show my tasks", "delete everything", "do something odd"]:
    tool, args = stand_in_model(sentence)
    print(f"{sentence!r:22} -> {tool} -> {run_tool('bob', tool, args)}")
print("\nlog:", [(r["tool"], r["ok"]) for r in LOG])

## Step 4: try to break it
Try: ask `run_tool("bob", "list_tasks", {})` and check that you cannot see Alice's tasks. Add a `rename_task` tool and its checks. Write down why the user id must never come from the model.

**Where this is in the project:** the task assistant services and `docs/evidence/day3.md` (5 of 5 runs, 24 invalid cases refused, confirmation before delete).